In [ ]:
!pip install -U tiktoken semantic-kernel==1.29.0 semantic-kernel[azure] python-dotenv azure-identity pydantic==2.9.2 --quiet


In [ ]:
# Azure AI Foundry Project
project_connection_string = "dummy"
model_deployment_name = "gpt-4o-mini"
azure_search_endpoint = "https://Dummy.search.windows.net/"
azure_search_api_key = "DummyKey"
azure_ai_search_index_name = "notebook-index"
azure_openai_api_version = "2023-05-15"
azure_openai_endpoint = "https://adummy-endpoint-name.openai.azure.com/"
azure_openai_api_key = "DummyKey"  
embedding_model_name = "text-embedding-ada-002"

In [4]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
import asyncio
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureTextEmbedding
from semantic_kernel.memory import VolatileMemoryStore, SemanticTextMemory
from semantic_kernel.functions import kernel_function
from typing import Annotated, List, Tuple
import uuid
import tiktoken

kernel = Kernel()
service_id = "azure_openai"

# Add Azure Chat Completion service
kernel.add_service(
    AzureChatCompletion(
        deployment_name=model_deployment_name,
        endpoint=azure_openai_endpoint,
        api_key=azure_openai_api_key,
        service_id=service_id
    )
)

# Configure function choice behavior
settings = kernel.get_prompt_execution_settings_from_service_id(service_id=service_id)
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()


In [5]:
from typing import Annotated
from semantic_kernel.functions import kernel_function
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import QueryType,VectorizableTextQuery, QueryCaptionType, QueryAnswerType

class SearchPlugin:
    """Plugin to interact with Azure AI Search."""

    def __init__(self, endpoint: str, index_name: str, api_key: str):
        self.search_client = SearchClient(
            endpoint=endpoint,
            index_name=index_name,
            credential=AzureKeyCredential(api_key)
        )

    @kernel_function(description="Search notebooks based on a query.")
    def search_notebooks(self, query: Annotated[str, "User's search query"]) -> Annotated[str, "Search results"]:
        
        vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=5, fields="notebook_name_vector,notebook_content_vector", exhaustive=True)
   
        results = self.search_client.search(search_text=query,
                        vector_queries=[vector_query],
                        select=["notebook_name", "notebook_content"],
                        query_type=QueryType.SEMANTIC,
                        semantic_configuration_name='notebook-semantic-config',
                        query_caption=QueryCaptionType.EXTRACTIVE,
                        query_answer=QueryAnswerType.EXTRACTIVE,
                        top=5)
        notebooks = [f"**Notebook Name**: {doc['notebook_name']}\n**Content**: {doc['notebook_content']}" for doc in results]
        return "\n\n".join(notebooks)


In [10]:
from semantic_kernel import Kernel
from semantic_kernel.functions import kernel_function

class CodeGenerationPlugin:
    """Plugin to generate code snippets using LLM."""

    def __init__(self, kernel: Kernel, memory: SemanticTextMemory):
        self.kernel = kernel
        self.memory = memory

    def chunk_context(self, context: str, max_tokens: int = 500) -> List[str]:
        """Chunk the context into approximately max_tokens tokens per chunk."""
        tokenizer = tiktoken.get_encoding("cl100k_base")  # Use OpenAI tokenizer
        tokens = tokenizer.encode(context)
        chunks = []

        # Split tokens into chunks of max_tokens
        for i in range(0, len(tokens), max_tokens):
            chunk_tokens = tokens[i:i + max_tokens]
            chunk_text = tokenizer.decode(chunk_tokens)
            chunks.append(chunk_text)

        return chunks

    async def store_chunks_in_memory(self, context: str):
        """Store the chunks in volatile memory."""
        chunks = self.chunk_context(context)
        for i, chunk in enumerate(chunks):
            # Use a unique key for each chunk
            memory_key = f"chunk_{i}"
            await self.memory.store(memory_key, chunk)

    async def retrieve_relevant_chunks(self, query: str, top_k: int = 3) -> List[str]:
        """Retrieve the top-k relevant chunks from memory."""
        results = await self.memory.search(query, top_k)
        return [result.value for result in results].join('\n')

    @kernel_function(description="Generate code based on context and user query.")
    async def generate_code(self, context: Annotated[str, "Context from search results"], query: Annotated[str, "User's query"]) -> Annotated[str, "Generated code"]:
        
        store_chunks_in_memory(context)
        final_context = retrieve_relevant_chunks(query)
        prompt = f"""You are a code generation assistant.

Based on the following context:
{final_context}

Generate code that addresses the following query:
{query}

Provide only the code snippet without additional explanations."""

        response = await self.kernel.invoke_prompt(prompt)
        return response.value


In [ ]:
# Initialize plugins
search_plugin = SearchPlugin(
    endpoint=azure_search_endpoint,
    index_name="notebook-index",
    api_key=azure_search_api_key
)

embedding_service = AzureTextEmbedding(
    deployment_name=embedding_model_name,
    endpoint=azure_openai_endpoint,
    api_key=azure_openai_api_key)
memory = SemanticTextMemory(storage=VolatileMemoryStore(), embeddings_generator=embedding_service)
code_generation_plugin = CodeGenerationPlugin(kernel, memory)

# Add plugins to the kernel
kernel.add_plugin(search_plugin, plugin_name="search")
kernel.add_plugin(code_generation_plugin, plugin_name="codegen")


In [12]:
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.functions import KernelArguments

# Define the User Helper Agent
user_helper_agent = ChatCompletionAgent(
    kernel=kernel,
    name="UserHelperAgent",
    instructions="""
You are an orchestrator agent that coordinates between SearchAgent and CodeHelperAgent to fulfill user requests.

- If the user requests relevant notebooks, use the search plugin to retrieve and return the top relevant notebooks.

- If the user requests code generation, first use the search plugin to find relevant notebook content, then use the codegen plugin to generate code based on the retrieved context and the user's query.

Present the final response to the user accordingly.
""",
    arguments=KernelArguments(settings=settings)
)


In [ ]:
import asyncio
from semantic_kernel.agents import ChatHistoryAgentThread

async def main():
    # Create a chat history thread
    thread = ChatHistoryAgentThread()

    # Simulate user input
    #user_input = "how to add 2 numbers?"
    user_input = "which notebook adds 2 numbers?"

    # Delegate the query to the user_helper_agent
    response = await user_helper_agent.get_response(
        messages=user_input,
        thread=thread
    )

    # Output the final response
    print(f"Final Response:\n{response.content}")

# Run the main function
await main()
